In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import i0
from scipy.signal import freqz
from IPython.display import display, HTML, Math
from ipywidgets import Output, HBox, Layout

# ============================================================
# PROBLEM 12.12.2 — KAISER-WINDOW FIR DESIGN
# ============================================================

plt.rcParams.update({'font.size':11.5,'axes.titlesize':13.0,'axes.labelsize':11.5,'xtick.labelsize':10.5,'ytick.labelsize':10.5,'legend.fontsize':10.0})

# ============================================================
# CSS
# ============================================================

display(HTML("""
<style>
.ex-root{width:970px;max-width:970px;font-family:Arial,sans-serif;}
.ex-header{background:linear-gradient(90deg,#7b1fa2,#ab47bc);color:white;padding:10px 15px;border-radius:8px 8px 0 0;font-size:18px;font-weight:bold;}
.ex-intro{background:#fbf7fc;border:1px solid #d7c4e2;border-top:none;padding:10px 13px;border-radius:0 0 8px 8px;font-size:13.5px;line-height:1.5;margin-bottom:10px;}
.ex-box{width:944px;border:1px solid #d7c4e2;border-radius:7px;padding:10px 12px;margin-bottom:10px;font-size:13.5px;line-height:1.55;}
.ex-title{font-weight:bold;color:#6a1b9a;font-size:14px;margin-bottom:7px;}
.ex-value{font-weight:bold;}
.ex-columns{display:flex;gap:40px;align-items:flex-start;}
.ex-column{flex:1;min-width:0;}
</style>
"""))

# ============================================================
# INTRODUCTION
# ============================================================

display(HTML("""
<div class="ex-root">
<div class="ex-header">Problem 12.12.2 — Kaiser-Window FIR Filter</div>
<div class="ex-intro">
The desired response is interpreted as the sum of a low-pass component of magnitude 1
and a high-pass component of magnitude 2. The Kaiser design equations are evaluated
numerically, the ideal impulse response is constructed and windowed, and the desired
and actual magnitude responses and FIR impulse response are plotted.
</div>
</div>
"""))

# ============================================================
# GIVEN SPECIFICATIONS
# ============================================================

wp1_n, ws1_n = 0.20, 0.30
ws2_n, wp2_n = 0.475, 0.525

ideal_c1_n, ideal_c2_n = 0.25, 0.50

wp1, ws1 = wp1_n*np.pi, ws1_n*np.pi
ws2, wp2 = ws2_n*np.pi, wp2_n*np.pi

Omega_s = 2*np.pi

display(HTML(f"""
<div class="ex-box">
<div class="ex-title">Given specifications</div>
<div class="ex-columns">

<div class="ex-column">
Real-filter band 1: <span class="ex-value">0.90 &lt; |H| &lt; 1.10</span><br>
Frequency range: <span class="ex-value">0 ≤ |ω| ≤ {wp1_n:.3f}π</span><br><br>
Real-filter stopband: <span class="ex-value">|H| &lt; 0.06</span><br>
Frequency range: <span class="ex-value">{ws1_n:.3f}π ≤ |ω| ≤ {ws2_n:.3f}π</span>
</div>

<div class="ex-column">
Real-filter band 2: <span class="ex-value">1.90 &lt; |H| &lt; 2.10</span><br>
Frequency range: <span class="ex-value">{wp2_n:.3f}π ≤ |ω| ≤ π</span><br><br>
Ideal levels: <span class="ex-value">1, 0, 2</span><br>
Ideal break frequencies: <span class="ex-value">{ideal_c1_n:.2f}π and {ideal_c2_n:.2f}π</span>
</div>

</div>
</div>
"""))

# ============================================================
# RIPPLE PARAMETER
# ============================================================

delta = 0.06/3
As = -20*np.log10(delta)

if As <= 21:
    alpha = 0.0
elif As <= 50:
    alpha = 0.5842*(As-21)**0.4 + 0.07886*(As-21)
else:
    alpha = 0.1102*(As-8.7)

# ============================================================
# TRANSITION WIDTHS AND CUTOFF FREQUENCIES
# ============================================================

dw1 = ws1-wp1
dw2 = wp2-ws2
dw = min(dw1,dw2)

dw1_n, dw2_n, dw_n = dw1/np.pi, dw2/np.pi, dw/np.pi

wc1 = ws1-dw/2
wc2 = ws2+dw/2

wc1_n, wc2_n = wc1/np.pi, wc2/np.pi

D = 0.9222 if As <= 21 else (As-7.95)/14.36
order_argument = Omega_s*D/dw
N = 1+int(order_argument)
delay = N/2

# ============================================================
# NUMERICAL EQUATIONS — TWO COLUMNS
# ============================================================

left_output = Output(layout=Layout(width='48%'))
right_output = Output(layout=Layout(width='48%'))

with left_output:
    display(Math(rf"\delta+2\delta\leq0.06\Rightarrow3\delta\leq0.06\Rightarrow\delta_{{max}}={delta:.2f}"))
    display(Math(rf"A_s=-20\log_{{10}}(\delta_{{max}})=-20\log_{{10}}({delta:.2f})={As:.6f}\,\mathrm{{dB}}"))
    display(Math(rf"\alpha=0.5842(A_s-21)^{{0.4}}+0.07886(A_s-21)={alpha:.6f}"))
    display(Math(rf"\Delta\omega_1={ws1_n:.3f}\pi-{wp1_n:.3f}\pi={dw1_n:.3f}\pi"))
    display(Math(rf"\Delta\omega_2={wp2_n:.3f}\pi-{ws2_n:.3f}\pi={dw2_n:.3f}\pi"))

with right_output:
    display(Math(rf"\Delta\omega=\min(\Delta\omega_1,\Delta\omega_2)={dw_n:.3f}\pi"))
    display(Math(rf"\omega_{{c1}}={ws1_n:.3f}\pi-\frac{{{dw_n:.3f}\pi}}{{2}}={wc1_n:.3f}\pi"))
    display(Math(rf"\omega_{{c2}}={ws2_n:.3f}\pi+\frac{{{dw_n:.3f}\pi}}{{2}}={wc2_n:.3f}\pi"))
    display(Math(rf"D=\frac{{A_s-7.95}}{{14.36}}={D:.6f}"))
    display(Math(rf"N\geq1+\left[\frac{{2\pi D}}{{\Delta\omega}}\right]=1+[\,{order_argument:.6f}\,]={N}"))

display(HBox([left_output,right_output],layout=Layout(width='970px',justify_content='space-between',align_items='flex-start')))

# ============================================================
# NUMERICAL SUMMARY
# ============================================================

display(HTML(f"""
<div class="ex-box">
<div class="ex-title">Numerical design result</div>
<div class="ex-columns">

<div class="ex-column">
δ = <span class="ex-value">{delta:.6f}</span><br>
A<sub>s</sub> = <span class="ex-value">{As:.6f} dB</span><br>
α = <span class="ex-value">{alpha:.6f}</span><br>
Δω<sub>1</sub> = <span class="ex-value">{dw1_n:.3f}π</span><br>
Δω<sub>2</sub> = <span class="ex-value">{dw2_n:.3f}π</span>
</div>

<div class="ex-column">
Governing Δω = <span class="ex-value">{dw_n:.3f}π</span><br>
ω<sub>c1</sub> = <span class="ex-value">{wc1_n:.3f}π</span><br>
ω<sub>c2</sub> = <span class="ex-value">{wc2_n:.3f}π</span><br>
D = <span class="ex-value">{D:.6f}</span><br>
Filter order N = <span class="ex-value">{N}</span>
</div>

</div>
</div>
"""))

# ============================================================
# IDEAL IMPULSE RESPONSE
# ============================================================

n = np.arange(N+1,dtype=float)
m = n-delay

hd = np.empty_like(m)
center = np.abs(m)<1e-12

hd[center] = wc1/np.pi + 2*(1-wc2/np.pi)
hd[~center] = (np.sin(wc1*m[~center])+2*np.sin(np.pi*m[~center])-2*np.sin(wc2*m[~center]))/(np.pi*m[~center])

display(HTML("""
<div class="ex-box">
<div class="ex-title">Ideal impulse response</div>
The ideal response consists of a low-pass component of magnitude 1 and a high-pass component of magnitude 2. Its delayed impulse response is
</div>
"""))

display(Math(rf"h_d[n]=\frac{{\sin[{wc1_n:.3f}\pi(n-{delay:.1f})]+2\sin[\pi(n-{delay:.1f})]-2\sin[{wc2_n:.3f}\pi(n-{delay:.1f})]}}{{\pi(n-{delay:.1f})}}"))

# ============================================================
# KAISER WINDOW AND ACTUAL FILTER
# ============================================================

x = (n-delay)/delay
w = i0(alpha*np.sqrt(np.maximum(0,1-x**2)))/i0(alpha)
h = hd*w

# ============================================================
# FREQUENCY RESPONSE
# ============================================================

omega,H = freqz(h,worN=8192)
omega_n = omega/np.pi
Hmag = np.abs(H)

# ============================================================
# NUMERICAL VERIFICATION
# ============================================================

band1_mask = omega_n <= wp1_n
stop_mask = (omega_n >= ws1_n) & (omega_n <= ws2_n)
band2_mask = omega_n >= wp2_n

band1_min, band1_max = np.min(Hmag[band1_mask]), np.max(Hmag[band1_mask])
stop_max = np.max(Hmag[stop_mask])
band2_min, band2_max = np.min(Hmag[band2_mask]), np.max(Hmag[band2_mask])

display(HTML(f"""
<div class="ex-box">
<div class="ex-title">Numerical verification of the resulting FIR filter</div>
<div class="ex-columns">

<div class="ex-column">
Band 1 minimum: <span class="ex-value">{band1_min:.6f}</span><br>
Band 1 maximum: <span class="ex-value">{band1_max:.6f}</span><br>
Stopband maximum: <span class="ex-value">{stop_max:.6f}</span>
</div>

<div class="ex-column">
Band 2 minimum: <span class="ex-value">{band2_min:.6f}</span><br>
Band 2 maximum: <span class="ex-value">{band2_max:.6f}</span><br>
Calculated N: <span class="ex-value">{N}</span>
</div>

</div>
</div>
"""))

# ============================================================
# FINAL FIGURES
# ============================================================

fig,(ax1,ax2) = plt.subplots(1,2,figsize=(13.5,4.6))

# ============================================================
# 1. MAGNITUDE RESPONSE
# ============================================================

ax1.plot(omega_n,Hmag,color='red',linewidth=1.7,label='Actual FIR response')

# Ideal response
ax1.plot([0,wc1_n],[1,1],'--',linewidth=1.2,label='Ideal response')
ax1.plot([wc1_n,wc1_n],[1,0],'--',linewidth=1.2)
ax1.plot([wc1_n,wc2_n],[0,0],'--',linewidth=1.2)
ax1.plot([wc2_n,wc2_n],[0,2],'--',linewidth=1.2)
ax1.plot([wc2_n,1],[2,2],'--',linewidth=1.2)

# Real-filter specification limits
ax1.plot([0,wp1_n],[0.90,0.90],':',linewidth=1.0,label='Specification limits')
ax1.plot([0,wp1_n],[1.10,1.10],':',linewidth=1.0)
ax1.plot([ws1_n,ws2_n],[0.06,0.06],':',linewidth=1.0)
ax1.plot([wp2_n,1],[1.90,1.90],':',linewidth=1.0)
ax1.plot([wp2_n,1],[2.10,2.10],':',linewidth=1.0)

ax1.axvline(wp1_n,linestyle=':',linewidth=0.9)
ax1.axvline(ws1_n,linestyle=':',linewidth=0.9)
ax1.axvline(ws2_n,linestyle=':',linewidth=0.9)
ax1.axvline(wp2_n,linestyle=':',linewidth=0.9)

ax1.axvspan(wp1_n,ws1_n,alpha=0.08)
ax1.axvspan(ws2_n,wp2_n,alpha=0.08)

ax1.set_xlim(0,1)
ax1.set_ylim(-0.08,2.20)
ax1.set_title('Magnitude Response and Design Specifications')
ax1.set_xlabel(r'Normalized frequency $\omega/\pi$')
ax1.set_ylabel(r'$|H(e^{j\omega})|$')
ax1.grid(True,linestyle=':',alpha=0.25)
ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=3,frameon=False)

# ============================================================
# 2. IMPULSE RESPONSE
# ============================================================

markerline,stemlines,baseline = ax2.stem(n,h,linefmt='r-',markerfmt='ro',basefmt=' ')
plt.setp(markerline,markersize=3.5)
plt.setp(stemlines,linewidth=1.0)

ax2.axvline(delay,linestyle='--',linewidth=1.1,label=rf'Delay = {delay:.1f}')
ax2.set_title('Kaiser-Window FIR Impulse Response')
ax2.set_xlabel('Sample index $n$')
ax2.set_ylabel('$h[n]$')
ax2.grid(True,linestyle=':',alpha=0.25)
ax2.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),frameon=False)

plt.subplots_adjust(left=0.07,right=0.98,top=0.90,bottom=0.23,wspace=0.30)
plt.show()
plt.close(fig)